# 2주차 실습 — 앞서 정한 배송 약속은 이후 주문에서도 지켜지는가

교재와 같은 Olist 주문 기록으로 주별 약속 후보를 평가해 보자. 앞선 주문으로 후보를 정한 뒤, 이후 주문에 그대로 적용해 실제 안내된 약속과 비교한다.

> **기존 약속을 주별 92분위수 후보로 바꾼다면, 안내하는 소요일과 지연율은 어떻게 달라지는가?**

약 60분 동안 진행한다. 준비 5분, 과제 1~4 각 5분, 과제 5~6 각 10분, 과제 7은 15분이다. 앞 과제의 결과를 다음 과제에서 이어서 쓴다. 각 과제의 코드와 검증 셀을 실행해 확인한 뒤 넘어간다.

교재에서 최근 배송 변화를 살펴본 2018년 6~8월을 평가 기간으로 둔다. 후보를 만들 때는 2017년 9월 이후 구매한 주문 중 2018년 6월 1일 전에 수령이 확인된 주문을 사용한다. 예를 들어 5월에 주문했더라도 6월에 받은 주문의 소요일은 6월 1일에 알 수 없으므로 후보 계산에서 제외한다.

0.92는 교재에서 가져온 비교 시나리오이며 이 랩에서 최적값을 찾는 것은 아니다. 교재가 전체 기간을 이미 살펴봤으므로 완전히 새로운 자료에 대한 독립 검증으로 해석하지 않는다. 이 실습에서는 후보의 소요일을 계산할 자료와 평가할 자료를 구분하는 데 집중한다.

**계산 과제 1~6은 변수 이름으로 확인한다.** 건수·기간은 정확히, 소요일은 0.01일, 비율은 0.0001의 허용 오차로 확인한다. 계산값은 그대로 두고 출력할 때만 반올림한다. 과제 7의 판단과 설명은 자동 채점하지 않으며 본인이 문장으로 작성해 제출한다.

## 준비 — 교재에서 계산한 주문 표

배송지 연결과 날짜 변환, 소요일 계산은 아래 코드로 제공한다. 실행하면 준비된 표가 `r`에 담긴다. 한 행은 주문 한 건이다.

| 열 | 뜻 |
|---|---|
| `order_purchase_timestamp` | 구매 시각 |
| `order_delivered_customer_date` | 고객이 받은 시각 |
| `customer_state` | 배송지의 주 |
| `actual_days` | 구매일부터 받은 날까지의 소요일 |
| `promised_days` | 구매일부터 실제 안내된 예상 배송일까지의 소요일 |
| `late` | 실제 안내된 예상 배송일을 넘겨 받았는지의 참·거짓 |

대상은 2017년 9월~2018년 8월의 구매 주문 중 배송 완료이며 수령 시각이 있는 주문이다. 날짜 차이는 교재처럼 주말·공휴일을 포함한다. 이후 비교도 배송 완료가 확인된 주문에 한정된다.

In [ ]:
# [셋업] 환경과 원자료 — 수정하지 않는다.
import os
import sys
import numpy as np
import pandas as pd

for _p in (".", "..", "../..", "../../.."):
    if os.path.exists(os.path.join(_p, "balab.py")):
        sys.path.insert(0, _p)
        break
else:
    !wget -q https://raw.githubusercontent.com/BALAB-PKNU/bizanalytics/main/balab.py

from balab import load, checker, summary
check = checker("week02")
olist = load("olist")
orders, customers = olist["orders"], olist["customers"]

df = orders.merge(customers, on="customer_id", how="left")
for column in ["order_purchase_timestamp", "order_delivered_customer_date", "order_estimated_delivery_date"]:
    df[column] = pd.to_datetime(df[column])
d = df[df["order_status"] == "delivered"].dropna(subset=["order_delivered_customer_date"]).copy()
DAY = pd.Timedelta(days=1)
d["actual_days"] = (d["order_delivered_customer_date"].dt.normalize() - d["order_purchase_timestamp"].dt.normalize()) / DAY
d["promised_days"] = (d["order_estimated_delivery_date"] - d["order_purchase_timestamp"].dt.normalize()) / DAY
d["late"] = d["actual_days"] > d["promised_days"]
r = d[d["order_purchase_timestamp"] >= "2017-09-01"]
r = r[r["order_purchase_timestamp"] < "2018-09-01"].copy()
r[["customer_state", "actual_days", "promised_days", "late"]].head()


## 과제 1 — 후보를 만들 주문과 평가할 주문을 나눈다

**앞선 주문과 이후 주문을 서로 다른 표에 담아 보자.**

`r`에서 구매일이 2018년 6월 1일 전인 주문을 `past`에 담고, 수령 시각도 그날 전인 주문만 남긴다. 구매일이 그날 이상인 주문은 `future`에 담는다. 두 표 모두 사본으로 만든다.

교재 6-9의 조건 필터와 `copy`를 사용한다. 각 표의 주문 수와 구매 시각의 최솟값·최댓값을 출력해 기간을 확인한다. 출력은 두 코드 셀에 나눠도 된다.

입력은 `r`, 만들 결과는 주문별 DataFrame인 `past`와 `future`다. 검증 셀은 주문 수와 구매 기간, 과거 표에 기준일 이후 수령 기록이 들어갔는지 확인한다.

In [ ]:
past = None  # TODO: 기준일 전에 구매하고 수령한 주문의 사본
future = None  # TODO: 기준일 이후에 구매한 주문의 사본
# 기준일은 2018-06-01이다 (조건 필터, copy, len, min, max)

In [ ]:
# TODO: future의 주문 수와 구매 시각의 최솟값·최댓값을 확인한다 (len, min, max)

In [ ]:
# [검증] 과제 1 — 수정하지 않는다.
if past is None or future is None:
    check(1, past_n=None)
else:
    check(1, past_n=len(past), future_n=len(future),
          past_start=str(past["order_purchase_timestamp"].min()), past_end=str(past["order_purchase_timestamp"].max()),
          future_start=str(future["order_purchase_timestamp"].min()), future_end=str(future["order_purchase_timestamp"].max()),
          unavailable=(past["order_delivered_customer_date"] >= "2018-06-01").sum())

## 과제 2 — 앞선 주문으로 주별 약속 후보를 만든다

**주마다 며칠을 약속할지 앞선 주문으로 계산해 보자.**

`past`의 실제 소요일을 주별로 묶어 0.92분위수를 구하고, 날짜로 안내할 수 있도록 올림해 `q92`에 담는다. 평가할 주문인 `future`는 이 계산에 쓰지 않는다.

교재 9-1의 `groupby`·`quantile`·`np.ceil`을 사용한다. 결과는 주 이름을 인덱스로 하는 Series다. 표를 출력해 지역별 후보 소요일을 확인한다. 검증 셀은 모든 주의 후보를 확인한다.

In [ ]:
q92 = None  # TODO: past에서 구한 주별 실제 소요일 0.92분위수를 올림한 Series

In [ ]:
# [검증] 과제 2 — 수정하지 않는다.
if q92 is None:
    check(2, n=None)
else:
    ordered = q92.sort_index()
    check(2, n=len(q92), a=(ordered.iloc[:12], 0.01), b=(ordered.iloc[12:24], 0.01), c=(ordered.iloc[24:], 0.01))

## 과제 3 — 평가 주문에 후보를 붙인다

**이후 주문이 각 배송지의 후보를 적용받도록 연결해 보자.**

`future`의 배송지에 `q92`를 대응시켜 새 열 `promise92`에 담는다. 후보는 과제 2에서 구한 값을 그대로 쓴다.

교재 8-1·9-2의 `map`을 사용한다. 배송지와 후보 열을 함께 출력하고, `isna`와 `sum`으로 후보가 붙지 않은 주문이 있는지 센다. 이 자료에서는 양쪽 기간에 모든 주가 있어 누락 없이 연결할 수 있다. 누락이 나오면 기간 선택과 연결을 확인한다.

입력은 `future`와 `q92`, 결과는 `future`에 추가한 숫자 열 `promise92`다. 검증 셀은 주문 수, 후보의 누락과 주별 대응값을 확인한다.

In [ ]:
# TODO: future의 배송지에 q92를 대응시켜 promise92 열을 추가한다 (map)
# 배송지·후보 표와 후보의 빈 값 수를 확인한다 (isna, sum)

In [ ]:
# [검증] 과제 3 — 수정하지 않는다.
if future is None or "promise92" not in future.columns:
    check(3, n=None)
else:
    mapped = future.groupby("customer_state")["promise92"].mean().sort_index()
    check(3, n=len(future), missing=future["promise92"].isna().sum(),
          a=(mapped.iloc[:12], 0.01), b=(mapped.iloc[12:24], 0.01), c=(mapped.iloc[24:], 0.01),
          max_spread=future.groupby("customer_state")["promise92"].std().max())

## 과제 4 — 후보에서 지연된 주문을 센다

**후보 소요일을 넘겨 받은 주문이 얼마나 되는지 세어 보자.**

`future`에서 실제 소요일이 후보 소요일보다 길면 참이 되도록 `late92` 열을 만든다. 같은 날에 받으면 지연이 아니다.

교재 6-5·8-3처럼 비교 연산으로 참·거짓을 만들고, `sum`으로 지연 건수와 `mean`으로 지연율을 확인한다. 결과는 `future`에 추가한 참·거짓 열 `late92`다. 검증 셀은 누락 여부와 지연 건수·비율을 확인한다.

In [ ]:
# TODO: future의 actual_days와 promise92를 비교해 late92 열을 만든다
# 지연 건수와 비율을 확인한다 (비교, sum, mean)

In [ ]:
# [검증] 과제 4 — 수정하지 않는다.
if future is None or "late92" not in future.columns:
    check(4, missing=None)
else:
    check(4, missing=future["late92"].isna().sum(), late_n=future["late92"].sum(), late_rate=(future["late92"].mean(), 0.0001))

## 과제 5 — 기존 약속과 후보를 전체에서 비교한다

**지연의 변화와 약속 소요일의 변화를 함께 비교해 보자.**

평가 기간의 같은 주문에서 기존 약속과 후보의 평균 소요일·지연율을 계산해 `comparison`에 담는다. 기존 약속은 `promised_days`·`late`, 후보는 `promise92`·`late92` 열에 있다.

교재 9-1·9-2처럼 사전으로 `DataFrame`을 만든다. `index`에는 행 이름 목록을 지정한다. 각 열은 주문별 값의 `mean`으로 구한다.

| 결과 표 | 이름과 뜻 |
|---|---|
| 행 이름 | `current` 기존 약속, `q92` 후보 |
| `mean_promise` 열 | 주문별 약속 소요일의 평균 |
| `late_rate` 열 | 주문별 지연 여부의 평균 |

결과는 2행 2열 표다. 출력할 때 소수 4자리로 반올림해 읽는다. 검증 셀은 두 행의 모든 값을 확인한다.

In [ ]:
comparison = None  # TODO: current·q92의 mean_promise·late_rate를 담은 비교표
# 평가 기간 future의 주문별 평균을 사용한다 (mean, DataFrame, round)

In [ ]:
# [검증] 과제 5 — 수정하지 않는다.
if comparison is None:
    check(5, n=None)
else:
    check(5, n=len(comparison), days=(comparison["mean_promise"], 0.01), rate=(comparison["late_rate"], 0.0001))

## 과제 6 — 주별로 같은 변화가 나타나는지 비교한다

**전체 결과와 다른 변화가 나타나는 지역을 찾아보자.**

`future`를 배송지로 묶고 아래 값을 집계해 `by_state`에 담는다. 교재 7-1의 `groupby`와 `agg`를 사용한다.

| 열 이름 | 집계할 값 |
|---|---|
| `orders` | 주문 수 |
| `promise_now` | 기존 약속 소요일 평균 |
| `promise92` | 후보 소요일 평균 |
| `late_now` | 기존 지연율 |
| `late92` | 후보 지연율 |

결과는 주 이름을 인덱스로 하는 표다. 기존 지연율이 높은 순서로 출력한다. `sort_values`의 `ascending`을 False로 둔다. 검증 셀은 모든 주의 주문 수와 두 약속·지연율을 확인한다.

In [ ]:
by_state = None  # TODO: future를 주별로 묶어 위 다섯 열을 집계한 표
# 주문 수는 size, 약속 소요일과 지연율은 mean으로 구한다 (groupby, agg)
# late_now가 큰 순서로 출력한다 (sort_values, round)

In [ ]:
# [검증] 과제 6 — 수정하지 않는다.
if by_state is None:
    check(6, n=None)
else:
    ordered = by_state.sort_index()
    check(6, n=len(by_state),
          orders_a=ordered["orders"].iloc[:12],
          orders_b=ordered["orders"].iloc[12:24],
          orders_c=ordered["orders"].iloc[24:],
          promise_now_a=(ordered["promise_now"].iloc[:12], 0.01),
          promise_now_b=(ordered["promise_now"].iloc[12:24], 0.01),
          promise_now_c=(ordered["promise_now"].iloc[24:], 0.01),
          promise92_a=(ordered["promise92"].iloc[:12], 0.01),
          promise92_b=(ordered["promise92"].iloc[12:24], 0.01),
          promise92_c=(ordered["promise92"].iloc[24:], 0.01),
          late_now_a=(ordered["late_now"].iloc[:12], 0.0001),
          late_now_b=(ordered["late_now"].iloc[12:24], 0.0001),
          late_now_c=(ordered["late_now"].iloc[24:], 0.0001),
          late92_a=(ordered["late92"].iloc[:12], 0.0001),
          late92_b=(ordered["late92"].iloc[12:24], 0.0001),
          late92_c=(ordered["late92"].iloc[24:], 0.0001))

## 과제 7 — 본인의 판단과 설명을 쓴다

**이 후보를 실제 약속에 적용할지, 자신의 판단을 설명해 보자.**

과제 5와 6의 표를 근거로 아래 내용을 이어서 작성한다. 적용·일부 지역 적용·보류 중 어느 판단도 가능하다. 계산 결과가 판단을 뒷받침해야 한다.

1. 전체에서 평균 약속 소요일과 지연율이 각각 얼마나 달라졌는지 수치로 설명한다.
2. 주별 변화가 다른 두 지역을 골라 주문 수와 기존·후보 값을 비교한다. 표본이 적은 지역이라면 그 점도 판단에 반영한다.
3. 적용 여부와 이유를 쓰고, 추가로 확인하고 싶은 정보 하나를 제시한다. 약속을 길게 안내하는 것과 배송 자체가 빨라지는 것을 구분한다.

이 비교는 배송이 완료된 주문의 결과다. 기준일에 아직 받지 못한 과거 주문은 후보 계산에서 빠졌고, 지역별 주문 수와 배송 상황도 기간에 따라 달라질 수 있다. 후보의 지연율이 꼭 0.08이어야 한다고 생각하거나 값을 맞추지 않는다.

**이 과제는 자동 채점하지 않는다.** 아래 마크다운 셀에 본인의 설명을 작성한다. 한 가지 정답 문장이나 특정 결론을 요구하지 않는다.

### 나의 판단과 설명

(이 셀을 편집해 전체 비교, 두 지역의 비교, 본인의 판단과 추가 확인 사항을 문장으로 작성한다.)

## 제출 전 확인

아래 완료 확인은 계산 과제 1~6의 상태다. 모두 통과했더라도 과제 7의 설명을 작성해야 제출이 끝난다. 위에서부터 실행하고, 본인의 계산과 설명이 담긴 노트북을 저장해 제출한다.

In [ ]:
# 계산 과제 1~6 완료 확인. 과제 7의 설명은 별도로 작성한다.
summary()